In [ ]:
%pip install "cmgdb>=1.3.2"

# A CMGDB primer

[CMGDB](https://github.com/marciogameiro/CMGDB) (Conley-Morse Graph Database)
is the engine behind every figure in this paper. Given a map and a rectangular
domain, it builds a rigorous outer-cover of the dynamics on a grid of boxes and
returns a **Morse graph**: the partial order of recurrent components, each
labelled with its Conley index.

This notebook is a standalone primer using the planar Leslie map directly --
no autoencoder, no `latentdynamics`. The example notebooks (01-05) drive this
same machinery on *learned* latent maps.

## The map and its box map

CMGDB needs two callables: the map `f` on points, and a **box map** `F` that
sends a rectangle to a rectangle enclosing the image of every point inside it.
Here we enclose the image by evaluating `f` at the four corners and taking the
componentwise min/max.

In [ ]:
import math

import CMGDB


def f(x):
    # Planar Leslie population map (paper section 5.1).
    th1, th2 = 20.0, 20.0
    return [(th1 * x[0] + th2 * x[1]) * math.exp(-0.1 * (x[0] + x[1])), 0.7 * x[0]]


def corners(rect):
    (x0_min, x1_min, x0_max, x1_max) = rect
    return [
        f([x0_min, x1_min]),
        f([x0_max, x1_min]),
        f([x0_min, x1_max]),
        f([x0_max, x1_max]),
    ]


def box_map(rect):
    ys = corners(rect)
    return [
        min(y[0] for y in ys),
        min(y[1] for y in ys),
        max(y[0] for y in ys),
        max(y[1] for y in ys),
    ]

## Compute the Morse graph

`CMGDB.Model` bundles the subdivision schedule, the domain bounds, and the box
map; `ComputeMorseGraph` runs the decomposition. A node with no outgoing edges
is *minimal* -- an attractor. We sweep the subdivision depth upward to find the
coarsest grid that already resolves both attractors of this map (its stable
invariant circle and its stable period-six orbit).

In [ ]:
subdiv_limit = 10000
lower_bounds = [-0.001, -0.001]
upper_bounds = [90.0, 70.0]

morse_graph = None
for subdiv in range(10, 21):
    model = CMGDB.Model(subdiv, subdiv, subdiv, subdiv_limit,
                        lower_bounds, upper_bounds, box_map)
    morse_graph, _ = CMGDB.ComputeMorseGraph(model)
    n_min = sum(1 for v in morse_graph.vertices() if len(morse_graph.adjacencies(v)) == 0)
    print(f"subdiv={subdiv}: {morse_graph.num_vertices()} vertices, {n_min} minimal node(s)")
    if n_min >= 2:
        print(f"\n-> two attractors resolved at subdiv={subdiv}")
        break

In [ ]:
CMGDB.PlotMorseGraph(morse_graph)

In [ ]:
CMGDB.PlotMorseSets(morse_graph)

## Padding the box map

Real (learned) maps are only known approximately, so the box map is usually
*padded*: each image rectangle is enlarged by the side lengths of its input box.
Padding makes the outer-cover conservative (it never undercounts recurrence) at
the cost of a coarser graph. The `latentdynamics` pipeline exposes this as the
`cmgdb.padding` config flag.

In [ ]:
def box_map_padded(rect):
    (x0_min, x1_min, x0_max, x1_max) = rect
    h0, h1 = x0_max - x0_min, x1_max - x1_min
    ys = corners(rect)
    return [
        min(y[0] for y in ys) - h0,
        min(y[1] for y in ys) - h1,
        max(y[0] for y in ys) + h0,
        max(y[1] for y in ys) + h1,
    ]


model = CMGDB.Model(20, 20, 20, subdiv_limit, lower_bounds, upper_bounds, box_map_padded)
morse_graph_padded, _ = CMGDB.ComputeMorseGraph(model)
CMGDB.PlotMorseGraph(morse_graph_padded)

In [ ]:
CMGDB.PlotMorseSets(morse_graph_padded)